In [ ]:
# M24CA1L306 Data Science Lab - Lab Exercise #21
# Student: Abhinav | MAC25MCA-2002 | S3 MCA (2025-27)
# Dataset: Iris.csv
# Task: EDA + K-Means clustering on Iris dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score


In [ ]:

# ─────────────────────────────────────────────
# Load dataset
# ─────────────────────────────────────────────
df = pd.read_csv("../Datasets/iris/iris.csv")
df.drop(columns=["Id"], inplace=True, errors="ignore")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst 5 rows:\n", df.head())
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDescriptive statistics:\n", df.describe())
print("\nSpecies distribution:\n", df["Species"].value_counts())

features = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]
X = df[features].values


In [ ]:

# ─────────────────────────────────────────────
# (i) EDA
# ─────────────────────────────────────────────
print("\nCorrelation matrix:\n", df[features].corr())


In [ ]:

# ─────────────────────────────────────────────
# (ii) Visualizations
# ─────────────────────────────────────────────
species_list = df["Species"].unique()
colors_map   = dict(zip(species_list, ["red", "blue", "green"]))

# Scatter plot
plt.figure(figsize=(7, 5))
for sp in species_list:
    sub = df[df["Species"] == sp]
    plt.scatter(sub["PetalLengthCm"], sub["PetalWidthCm"],
                label=sp, alpha=0.7, color=colors_map[sp])
plt.xlabel("Petal Length (cm)")
plt.ylabel("Petal Width (cm)")
plt.title("Scatter Plot - Petal Length vs Petal Width (by Species)")
plt.legend()
plt.tight_layout()
plt.savefig("../Outputs/lab-21_Output/lab21_iris_scatter.png", dpi=100)
plt.show()

# Histogram
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, feat in zip(axes.flatten(), features):
    for sp in species_list:
        ax.hist(df[df["Species"] == sp][feat], alpha=0.6, label=sp, bins=15)
    ax.set_title(feat)
    ax.legend(fontsize=7)
plt.suptitle("Histograms - Iris Features by Species")
plt.tight_layout()
plt.savefig("../Outputs/lab-21_Output/lab21_iris_histogram.png", dpi=100)
plt.show()

# Box plot
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, feat in zip(axes.flatten(), features):
    df.boxplot(column=feat, by="Species", ax=ax)
    ax.set_title(feat)
plt.suptitle("Box Plots - Iris Features by Species")
plt.tight_layout()
plt.savefig("../Outputs/lab-21_Output/lab21_iris_boxplot.png", dpi=100)
plt.show()


In [ ]:

# ─────────────────────────────────────────────
# (iii) K-Means with different k values
# ─────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias = []
k_range  = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, marker="o", color="steelblue")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow Graph - Iris (K-Means)")
plt.tight_layout()
plt.savefig("../Outputs/lab-21_Output/lab21_iris_elbow.png", dpi=100)
plt.show()

print("\nk  | Silhouette Score | Davies-Bouldin Index")
print("-" * 45)
for k in [2, 3, 4]:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil    = silhouette_score(X_scaled, labels)
    db     = davies_bouldin_score(X_scaled, labels)
    print(f"k={k} | {sil:.4f}           | {db:.4f}")


In [ ]:

# ─────────────────────────────────────────────
# (iv) Silhouette Score and Davies-Bouldin Index (k=3)
# ─────────────────────────────────────────────
km3    = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = km3.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, labels)
db  = davies_bouldin_score(X_scaled, labels)
print(f"\nk=3  |  Silhouette Score: {sil:.4f}  |  Davies-Bouldin Index: {db:.4f}")

plt.figure(figsize=(8, 5))
for cluster in range(3):
    mask = labels == cluster
    plt.scatter(X[mask, 2], X[mask, 3], label=f"Cluster {cluster}", alpha=0.7)
plt.xlabel("Petal Length (cm)")
plt.ylabel("Petal Width (cm)")
plt.title("K-Means Clusters (k=3) - Iris")
plt.legend()
plt.tight_layout()
plt.savefig("../Outputs/lab-21_Output/lab21_iris_clusters.png", dpi=100)
plt.show()

print("\nAll tasks complete. Plots saved.")